# NutriFlow assistant — free-GPU fine-tune (Colab)

Trains the tool-calling assistant on a **free Colab T4 GPU ($0)** and exports a **GGUF** you drop straight into LM Studio. Nothing runs on your PC.

## How to run
1. **Runtime → Change runtime type → T4 GPU** (free tier is fine).
2. **Runtime → Run all.**
3. When the upload cell appears, upload **`data/finetune.jsonl`** from the project (452 training examples).
4. Wait ~30–50 min. The last cell downloads **`nutriflow-assistant.Q4_K_M.gguf`**.
5. Back on your PC: put that file in LM Studio's models folder, load it, Start Server on :1234, and set `LOCAL_AI_MODEL=nutriflow-assistant` in `.env.local`.

Training happens off your machine, so the driver/power crashes can't occur. Local **inference** of the result is light and safe (same as your current 7B).

### 1. Install Unsloth (fast QLoRA + built-in GGUF export)

In [ ]:
%%capture
!pip install unsloth
# If the line above ever errors on Colab's preinstalled torch, use instead:
# !pip install --no-deps "xformers<0.0.27" "trl<0.12" peft accelerate bitsandbytes
# !pip install --no-deps unsloth

### 2. Config — which base model to fine-tune
`7B` = best quality (matches the model your app already runs). If training is too slow or hits memory limits on the free T4, change `BASE` to the 3B line — still very good for this narrow task, ~2x faster.

In [ ]:
BASE = "unsloth/Qwen2.5-7B-Instruct"      # best quality
# BASE = "unsloth/Qwen2.5-3B-Instruct"    # faster / lighter alternative
MAX_SEQ = 2048
EPOCHS = 3
OUT_NAME = "nutriflow-assistant"
print("Base:", BASE)

### 3. Upload the dataset (`data/finetune.jsonl`)

In [ ]:
from google.colab import files
up = files.upload()   # pick data/finetune.jsonl
DATA = list(up.keys())[0]
print("Uploaded:", DATA)
print("Lines:", sum(1 for _ in open(DATA, encoding="utf-8")))

### 4. Load base model in 4-bit + attach LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    dtype=None,   # auto (fp16 on T4)
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=32, lora_dropout=0.0, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                     "gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("LoRA attached.")

### 5. Build the training text (apply the chat template)

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files=DATA, split="train")

def to_text(ex):
    return {"text": tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False)}

ds = ds.map(to_text, remove_columns=ds.column_names)
print(ds[0]["text"][:600])

### 6. Train — loss only on the assistant's tool-call JSON
`train_on_responses_only` masks the system prompt + user message, so the model learns to *produce* the JSON, not memorize the prompt.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=EPOCHS,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        optim="adamw_8bit",
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="outputs",
        report_to=[],
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

trainer.train()

### 7. Quick sanity check — does it emit valid tool-call JSON?

In [ ]:
FastLanguageModel.for_inference(model)
msgs = [
    {"role": "system", "content": "You are the meal-plan assistant. Output JSON with a 'reply' and a list of 'operations'."},
    {"role": "user", "content": "make it cheaper and no onions this week"},
]
inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.0)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

### 8. Export GGUF (Q4_K_M) and download

In [ ]:
model.save_pretrained_gguf(OUT_NAME, tokenizer, quantization_method="q4_k_m")

In [ ]:
import glob, os
from google.colab import files
ggufs = glob.glob(f"{OUT_NAME}/**/*.gguf", recursive=True) + glob.glob("*.gguf")
ggufs = sorted(set(ggufs), key=os.path.getsize, reverse=True)
print("Found GGUF files:", ggufs)
assert ggufs, "No GGUF produced — check the export cell output."
files.download(ggufs[0])